# Row-wise softmax
Run from this directory. Set `SHAPE` to the workload and `AUTOTUNE = True` to time candidate `(block_size, threads)` pairs once per shape, dtype and GPU. Rows wider than the selected block use the online path. Set `CONFIG` to a pair to test a specific configuration instead.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in Path.cwd().parents if (p / 'common').is_dir())
sys.path.insert(0, str(ROOT))

import torch
from common.asm import dump_asm
from common.inputs import make_inputs
from common.run import make_runner, run_benchmark
from cute_softmax import softmax_wide
from tilelang_softmax import softmax_kernel_, specialize, tune_softmax
from torch_softmax import softmax_reference, torch_softmax_wide

DEVICE = torch.device('cuda:0')  # Use CUDA_VISIBLE_DEVICES to select the GPU.
torch.cuda.set_device(DEVICE)
SHAPE = (4096, 4096)
AUTOTUNE = False
CONFIG = None  # e.g. (1024, 128) to force the online path at this shape

In [ ]:
def compile_tilelang(kernel, x):
    if CONFIG is not None:
        return kernel(*x.shape, *CONFIG, 'float32')
    return specialize(kernel, x, autotune=AUTOTUNE)

impls = {
    'torch': make_runner('torch', softmax_reference),
    'torch_compile': make_runner('torch_compile', softmax_reference),
    'torch_wide': make_runner('torch', torch_softmax_wide),
    'cute_wide': make_runner('cute', softmax_wide(autotune=False)),
    'tilelang': make_runner('tilelang', softmax_kernel_, specialize=compile_tilelang, name='softmax'),
}
inputs = make_inputs(SHAPE, dtype=torch.float32, seed=42, device=DEVICE)
if AUTOTUNE and CONFIG is None:
    print('Tuned (block_size, threads):', tune_softmax(*inputs))
results = run_benchmark(impls, inputs, reference=softmax_reference, flops=5 * inputs[0].numel())

In [ ]:
for name, runner in impls.items():
    dump_asm(runner.asm(), Path('asm') / name)

In [ ]:
# Export the generated CUDA source for the benchmark's TileLang kernel.
cuda_dir = Path('cuda') / 'tilelang'
cuda_dir.mkdir(parents=True, exist_ok=True)
source = compile_tilelang(softmax_kernel_, inputs[0]).get_kernel_source()
(cuda_dir / 'softmax.cu').write_text(source, encoding='utf-8')